# SmolVLA Ablation: Action Chunk Size

# SmolVLA on LIBERO-Spatial

**Purpose:** Train and evaluate two chunk_size variants (10, 25) against the baseline (50).
This ablation is especially significant: the reference model uses n_action_steps=1
(closed-loop, ~123 forward passes/episode). Our variants probe the spectrum from
open-loop (chunk=50, ~3 passes) toward closed-loop (chunk=10, ~12 passes).

**Runtime:** H100 80GB. Estimated ~2-2.5 hours total.

**All hyperparameters identical to baseline except `chunk_size`:**
- steps=20000, batch_size=2, seed=42, save_freq=5000, use_amp=true

| Variant | chunk_size | Output dim | Fwd passes/ep (~123 steps) |
|---------|-----------|------------|----------------------------|
| Reference | 50 (n_action_steps=1) | 1600D | ~123 (closed-loop) |
| Baseline (done) | 50 | 1600D | ~3 (open-loop) |
| Variant A | 10 | 320D | ~12 |
| Variant B | 25 | 800D | ~5 |

## 0. Setup

In [ ]:
# Mount Google Drive for checkpoint backup
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install dependencies
!pip install "lerobot[libero] @ git+https://github.com/huggingface/lerobot.git" -q
!pip install num2words -q

In [ ]:
# Fix egl_probe cmake version issue
!git clone https://github.com/StanfordVL/egl_probe.git /tmp/egl_probe
!cd /tmp/egl_probe && sed -i 's/cmake_minimum_required(VERSION 2.8.12)/cmake_minimum_required(VERSION 3.10)/' egl_probe/CMakeLists.txt
!pip install /tmp/egl_probe -q

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Environment variables set.')

In [ ]:
# Wandb login
!wandb login

In [ ]:
# Create Artifact directoryectories
import os
dirs = [
    '/content/artifacts/checkpoints/ablation_chunk10',
    '/content/artifacts/checkpoints/ablation_chunk25',
    '/content/artifacts/eval_chunk10',
    '/content/artifacts/eval_chunk25',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'Ready: {d}')

In [ ]:
# Quick MuJoCo EGL rendering check
import mujoco
print(f'MuJoCo version: {mujoco.__version__}')
print(f'MUJOCO_GL = {os.environ.get("MUJOCO_GL", "NOT SET")}')

---
## 1. Ablation A: chunk_size=10

### 1.1 Train chunk_size=10

In [ ]:
%%time
!lerobot-train \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --batch_size=2 \
  --steps=20000 \
  --eval_freq=0 \
  --save_freq=5000 \
  --policy.use_amp=true \
  --policy.chunk_size=10 \
  --policy.n_action_steps=10 \
  --wandb.enable=true \
  --wandb.project=smolvla-libero \
  --output_dir=/content/outputs/train/ablation_chunk10 \
  --seed=42

In [ ]:
# Backup chunk10 checkpoints to Drive
!cp -r /content/outputs/train/ablation_chunk10/checkpoints/* \
  /content/artifacts/checkpoints/ablation_chunk10/
print('chunk10 checkpoints backed up to Drive.')
!ls -lh /content/artifacts/checkpoints/ablation_chunk10/

In [ ]:
# Verify chunk_size and n_action_steps in saved config
import json

config_path = '/content/outputs/train/ablation_chunk10/checkpoints/020000/pretrained_model/config.json'
try:
    with open(config_path) as f:
        cfg = json.load(f)
    cs = cfg.get('chunk_size', 'NOT FOUND')
    nas = cfg.get('n_action_steps', 'NOT FOUND')
    print(f'chunk_size = {cs}  (expected: 10)')
    print(f'n_action_steps = {nas}  (expected: 10)')
    if nas != 10:
        print('WARNING: n_action_steps != chunk_size. Eval may execute wrong number of steps.')
        print('If n_action_steps=50 (inherited from base), the model will try to execute')
        print('50 actions from a 10-step prediction -- likely causing errors or zero-padding.')
        print('Check if lerobot-eval respects the checkpoint config or needs --policy.n_action_steps=10')
except FileNotFoundError:
    # Config might be nested differently
    import glob
    candidates = glob.glob('/content/outputs/train/ablation_chunk10/checkpoints/020000/**/*.json', recursive=True)
    print(f'config.json not found at expected path. Found JSON files:')
    for c in candidates:
        print(f'  {c}')

### 1.2 Evaluate chunk_size=10

Output redirected to Drive log file to prevent notebook page crash.

In [ ]:
%%time
# Eval chunk10
!echo "N" | lerobot-eval \
  --policy.path=/content/outputs/train/ablation_chunk10/checkpoints/020000/pretrained_model \
  --env.type=libero \
  --env.task=libero_spatial \
  --eval.batch_size=3 \
  --eval.n_episodes=3 \
  --eval.use_async_envs=false \
  --policy.use_amp=true \
  --output_dir=/content/outputs/eval/ablation_chunk10 \
  --seed=42 \
  > /content/artifacts/eval_chunk10_log.txt 2>&1

print('chunk10 eval complete. Check log at /content/artifacts/eval_chunk10_log.txt')

In [ ]:
# Show last 30 lines of eval log
!head -70 /content/artifacts/eval_chunk10_log.txt

In [ ]:
# Backup chunk10 eval results and videos to Drive
!cp -r /content/outputs/eval/ablation_chunk10/* \
  /content/artifacts/eval_chunk10/
print('chunk10 eval results backed up to Drive.')
!find /content/artifacts/eval_chunk10/ -name '*.mp4' | head -5
!find /content/artifacts/eval_chunk10/ -name '*.json' | head -5

---
## 2. Ablation B: chunk_size=25

### 2.1 Train chunk_size=25

In [ ]:
%%time
!lerobot-train \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --batch_size=2 \
  --steps=20000 \
  --eval_freq=0 \
  --save_freq=5000 \
  --policy.use_amp=true \
  --policy.chunk_size=25 \
  --policy.n_action_steps=25 \
  --wandb.enable=true \
  --wandb.project=smolvla-libero \
  --output_dir=/content/outputs/train/ablation_chunk25 \
  --seed=42

In [ ]:
# Backup only chunk25 final checkpoint
!mkdir -p /content/artifacts/checkpoints/ablation_chunk25/020000
!cp -r /content/outputs/train/ablation_chunk25/checkpoints/020000/* \
  /content/artifacts/checkpoints/ablation_chunk25/020000/
print('chunk25 020000 backed up.')

In [ ]:
# Verify chunk_size and n_action_steps in saved config
import json

config_path = '/content/outputs/train/ablation_chunk25/checkpoints/020000/pretrained_model/config.json'
try:
    with open(config_path) as f:
        cfg = json.load(f)
    cs = cfg.get('chunk_size', 'NOT FOUND')
    nas = cfg.get('n_action_steps', 'NOT FOUND')
    print(f'chunk_size = {cs}  (expected: 25)')
    print(f'n_action_steps = {nas}  (expected: 25)')
    if nas != 25:
        print('WARNING: n_action_steps != chunk_size. Eval may execute wrong number of steps.')
        print('If n_action_steps=50 (inherited from base), the model will try to execute')
        print('50 actions from a 25-step prediction -- likely causing errors or zero-padding.')
        print('Check if lerobot-eval respects the checkpoint config or needs --policy.n_action_steps=25')
except FileNotFoundError:
    import glob
    candidates = glob.glob('/content/outputs/train/ablation_chunk25/checkpoints/020000/**/*.json', recursive=True)
    print(f'config.json not found at expected path. Found JSON files:')
    for c in candidates:
        print(f'  {c}')

### 2.2 Evaluate chunk_size=25

In [ ]:
%%time
# Eval chunk25
!echo "N" | lerobot-eval \
  --policy.path=/content/outputs/train/ablation_chunk25/checkpoints/020000/pretrained_model \
  --env.type=libero \
  --env.task=libero_spatial \
  --eval.batch_size=3 \
  --eval.n_episodes=3 \
  --eval.use_async_envs=false \
  --policy.use_amp=true \
  --output_dir=/content/outputs/eval/ablation_chunk25 \
  --seed=42 \
  > /content/artifacts/eval_chunk25_log.txt 2>&1

print('chunk25 eval complete. Check log at /content/artifacts/eval_chunk25_log.txt')

In [ ]:
# Show last 30 lines of eval log
!tail -30 /content/artifacts/eval_chunk25_log.txt

In [ ]:
# Backup chunk25 eval results and videos to Drive
!cp -r /content/outputs/eval/ablation_chunk25/* \
  /content/artifacts/eval_chunk25/
print('chunk25 eval results backed up to Drive.')
!find /content/artifacts/eval_chunk25/ -name '*.mp4' | head -5
!find /content/artifacts/eval_chunk25/ -name '*.json' | head -5

---
## 3. Summary & Sanity Check

In [ ]:
import json
from pathlib import Path

print('=' * 60)
print('ABLATION SUMMARY: Chunk Size')
print('=' * 60)

# Check for eval results JSONs
for variant in ['chunk10', 'chunk25']:
    eval_dir = Path(f'/content/artifacts/eval_{variant}')
    json_files = list(eval_dir.rglob('*.json'))
    mp4_files = list(eval_dir.rglob('*.mp4'))
    print(f'\n--- {variant} ---')
    print(f'  JSON files: {len(json_files)}')
    print(f'  Video files: {len(mp4_files)}')
    for jf in json_files:
        print(f'  {jf.name}')
        try:
            data = json.loads(jf.read_text())
            # Try to find success rate info
            if 'per_task_success_rate' in data:
                print(f'    Per-task: {data["per_task_success_rate"]}')
                print(f'    Aggregate: {data.get("aggregate_success_rate", "N/A")}')
            else:
                # Print top-level keys so we can find the data
                print(f'    Keys: {list(data.keys())[:10]}')
        except Exception as e:
            print(f'    Could not parse: {e}')

print('\n' + '=' * 60)
print('Drive backup locations:')
print('  Checkpoints: /content/artifacts/checkpoints/ablation_chunk{10,25}/')
print('  Eval results: /content/artifacts/eval_chunk{10,25}/')
print('  Eval logs: /content/artifacts/eval_chunk{10,25}_log.txt')
print('=' * 60)

In [ ]:
# Check Wandb runs were logged
!wandb runs --project smolvla-libero 2>/dev/null || echo 'Use Wandb dashboard to verify runs manually.'

---
## 4. Post-Run: Extract Results for Report

Run this after both variants complete to extract per-task success rates.
The eval output structure may vary -- adapt the parsing below to match.

In [ ]:
# Parse eval logs for success rate lines
for variant in ['chunk10', 'chunk25']:
    log_path = f'/content/artifacts/eval_{variant}_log.txt'
    print(f'\n=== {variant} eval log (success-related lines) ===')
    try:
        with open(log_path) as f:
            for line in f:
                line_lower = line.lower()
                if any(kw in line_lower for kw in ['success', 'rate', 'task', 'avg', 'mean', 'episode', 'result']):
                    print(line.rstrip())
    except FileNotFoundError:
        print(f'  Log not found at {log_path}')

In [ ]:
# Experiment 1: Decoupling chunk_size from n_action_steps
# Same baseline checkpoint (chunk50, 20K steps), but execute only 5 actions before replanning
%%time
!echo "N" | lerobot-eval \
  --policy.path=/content/artifacts/checkpoints/baseline/020000/pretrained_model \
  --policy.n_action_steps=5 \
  --env.type=libero \
  --env.task=libero_spatial \
  --eval.batch_size=3 \
  --eval.n_episodes=3 \
  --eval.use_async_envs=false \
  --policy.use_amp=true \
  --output_dir=/content/outputs/eval/ablation_nsteps5 \
  --seed=42 \
  > /content/artifacts/eval_nsteps5_log.txt 2>&1
print("NSTEPS5 EVAL DONE")

In [ ]:
# Also check the eval output directory structure
for variant in ['chunk10', 'chunk25']:
    print(f'\n=== /content/outputs/eval/ablation_{variant} ===')
    !find /content/outputs/eval/ablation_{variant} -type f | head -20